<a href="https://colab.research.google.com/github/HISATAKA-KATO/EU_M_Math-Repository/blob/main/Chap08_Ex_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#データ加工・処理・分析ライブラリ
import numpy as np
import numpy.random as random
import scipy as sp
import pandas as pd
from pandas import Series, DataFrame

#可視化ライブラリ
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

#機械学習ライブラリ
import sklearn

#少数第3位まで表示
%precision 3

#インポート
import requests
import io
import pandas as pd

In [2]:
#自動車価格データを取得
ur1 = 'https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(ur1).content

#取得したデータをデータフレームオブジェクトとして読み込む
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

#データの列にラベルを設定
auto.columns = ['symboling','normalized-losses','make','fuel-type','aspiration','num-of-doors',
                'body-style','drive-wheels','engine-location','wheel-base','length','width',
                'height','curb-weight','engine-type','num-of-cylinders','engine-size',
                'fuel-system','bore','stroke','compression-ratio','horsepower','peak-rpm',
                'city-mpg','highway-mpg','price']

#それぞれのカラムに？が何個あるのかカウント
print('自動車データ形式:{}\n'.format(auto.shape))
auto.head()
auto = auto[['price','horsepower','width','height']]
auto.isin(['?']).sum()
print(auto.isin(['?']).sum())

#？をNaNに置換して、NaNがある行を削除
auto = auto.replace('?', np.nan).dropna()
print('\n自動車データの形式:{}\n'.format(auto.shape))

print('データの型確認(型変換前) \n{}\n'.format(auto.dtypes))

auto = auto.assign(price=pd.to_numeric(auto.price))
auto = auto.assign(horsepower=pd.to_numeric(auto.horsepower))
print('データの型確認(型変換後) \n{}'.format(auto.dtypes))

#相関の確認
auto.corr()

自動車データ形式:(205, 26)

price         4
horsepower    2
width         0
height        0
dtype: int64

自動車データの形式:(199, 4)

データの型確認(型変換前) 
price          object
horsepower     object
width         float64
height        float64
dtype: object

データの型確認(型変換後) 
price           int64
horsepower      int64
width         float64
height        float64
dtype: object


,price,horsepower,width,height
price,1.000000,0.810533,0.753871,0.134990
horsepower,0.810533,1.000000,0.615315,-0.087407
width,0.753871,0.615315,1.000000,0.309223
height,0.134990,-0.087407,0.309223,1.000000


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# データの再読み込みと前処理（engine-sizeを含めるため）
res = requests.get(ur1).content
auto_full = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None, na_values='?')
auto_full.columns = ['symboling','normalized-losses','make','fuel-type','aspiration','num-of-doors',
                'body-style','drive-wheels','engine-location','wheel-base','length','width',
                'height','curb-weight','engine-type','num-of-cylinders','engine-size',
                'fuel-system','bore','stroke','compression-ratio','horsepower','peak-rpm',
                'city-mpg','highway-mpg','price']

# 必要な列を抽出して欠損値を削除
sub_data = auto_full[['price', 'width', 'engine-size']].dropna()

# 説明変数(X)と目的変数(y)
X = sub_data[['width', 'engine-size']]
y = sub_data['price']

# 訓練データとテストデータに分割 (50%ずつ、random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

# モデルの構築(重回帰クラスの初期化と学習)
model = LinearRegression()
model.fit(X_train, y_train)

# スコアの算出
print('決定係数(train): {:.3f}'.format(model.score(X_train, y_train)))
print('決定係数(test): {:.3f}'.format(model.score(X_test, y_test)))

決定係数(train): 0.783
決定係数(test): 0.778


In [7]:
#リッジ回帰用のクラス
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

#訓練データとテストデータ分割
x = auto.drop('price', axis=1)
y = auto['price']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.5, random_state=0)

#モデルの構築と評価
linear = LinearRegression()
ridge = Ridge(random_state=0)

for model in [linear, ridge]:
    model.fit(x_train, y_train)
    print('{}(train):{:.6f}'.format(model.__class__.__name__, model.score(x_train, y_train)))
    print('{}(test):{:.6f}'.format(model.__class__.__name__, model.score(x_test, y_test)))

LinearRegression(train):0.733358
LinearRegression(test):0.737069
Ridge(train):0.733355
Ridge(test):0.737768


In [8]:
# リッジ回帰・ラッソ回帰用のクラスをインポート
from sklearn.linear_model import Ridge, Lasso

# モデルのリストを定義（通常の重回帰、リッジ、ラッソ）
linear = LinearRegression()
ridge = Ridge(random_state=0)
lasso = Lasso(random_state=0)

models = [linear, ridge, lasso]

for model in models:
    model.fit(x_train, y_train)
    print('{}(train): {:.6f}'.format(model.__class__.__name__, model.score(x_train, y_train)))
    print('{}(test): {:.6f}'.format(model.__class__.__name__, model.score(x_test, y_test)))

LinearRegression(train): 0.733358
LinearRegression(test): 0.737069
Ridge(train): 0.733355
Ridge(test): 0.737768
Lasso(train): 0.733358
Lasso(test): 0.737107


In [9]:
# 異なるalpha(教科書ではλ？)値でのラッソ回帰を比較
alphas = [0.1, 1.0, 10.0, 100.0]

for a in alphas:
    lasso_model = Lasso(alpha=a, random_state=0)
    lasso_model.fit(x_train, y_train)

    print(f'Lasso(alpha={a}):')
    print(f'  決定係数(train): {lasso_model.score(x_train, y_train):.6f}')
    print(f'  決定係数(test) : {lasso_model.score(x_test, y_test):.6f}')
    # 係数が0になった数を確認
    zero_coeffs = np.sum(lasso_model.coef_ == 0)
    print(f'  係数が0の数: {zero_coeffs} / {len(lasso_model.coef_)}')
    print('-' * 30)

Lasso(alpha=0.1):
  決定係数(train): 0.733358
  決定係数(test) : 0.737073
  係数が0の数: 0 / 3
------------------------------
Lasso(alpha=1.0):
  決定係数(train): 0.733358
  決定係数(test) : 0.737107
  係数が0の数: 0 / 3
------------------------------
Lasso(alpha=10.0):
  決定係数(train): 0.733357
  決定係数(test) : 0.737416
  係数が0の数: 0 / 3
------------------------------
Lasso(alpha=100.0):
  決定係数(train): 0.733288
  決定係数(test) : 0.740234
  係数が0の数: 0 / 3
------------------------------
